In [ ]:
# For the ai's data, I will be using pandas
import pandas as pd

In [ ]:
# Read the csv into a dataframe
df = pd.read_csv("training_data.csv")

# df

In [ ]:
# df.iloc[0:3]
# df.head()
# df.tail()

The below code gets a list of all rows where the value 'Risk' is set to "good" or "bad", and then prints the length of that list. There are both an even amount of 'good' rows and 'bad' rows in these data.

In [67]:
# risk["Age"].count()
risk_good = df[df["Risk"] == "good"]
risk_bad = df[df["Risk"] == "bad"]
print(len(risk_good["Age"]))
print(len(risk_bad["Age"]))


560
560


Additionally, we can print out the number of times a specific field appears within the dataframe.
In the below example, we print the length of the rows in the dataframe for the four possible values of the 'Job' field. In this case, 0, 1, 2, and 3.
Job type 2 is the most common.

In [11]:
print(len(df[df["Job"] == 0]))
print(len(df[df["Job"] == 1]))
print(len(df[df["Job"] == 2]))
print(len(df[df["Job"] == 3]))

20
238
686
176


The below code is a demo of the type of rule that my AI will be using.

It gets all fields in the dataframe for job, and calculates how many rows have that job with a risk of 'good' or 'bad'.
We take (total good risks / total bad risks) and use that as our 'score'. This score system is refined later on.


In [ ]:
key_job = {}

# Loop over all possible fields of 'Job' in the df
# i can be 1, 2, 3, 4
for i in df["Job"].unique():
    # job_type is set to all jobs of type i (job type = dataframe where "Job" is (0, 1, 2, or 3))
    job_type = df["Job"] == i

    # good and bad become subsets of the dataframe where the job_type has a Risk of good or bad
    good = df[( job_type) & (df["Risk"] == "good")]
    bad = df[( job_type) & (df["Risk"] == "bad")]

    # Rudimentary old scoring system
    score = -1 + len(good) / len(bad)

    print(f"Total with job [{i}]:  {len(df[ job_type])}")
    print(f"    Good: {len(good)}")
    print(f"    Bad: {len(bad)}")
    print(f"    Points for job: {score:+.5f}")

    # Storing score if it were to be used later.
    key_job[i] = score

Total with job [1]:  238
    Good: 124
    Bad: 114
    Points for job: +0.08772
Total with job [2]:  686
    Good: 340
    Bad: 346
    Points for job: -0.01734
Total with job [3]:  176
    Good: 84
    Bad: 92
    Points for job: -0.08696
Total with job [0]:  20
    Good: 12
    Bad: 8
    Points for job: +0.50000


Old code that can be ignored.

In [ ]:
# keys = {}

# for column in df.columns:
#     if column == "ID" or column == "Risk": continue
#     for option in df[column].unique():
#         # print(f"Column: {column} Option: {option}\n")
#         this_type = df[column] == option;
#         good = df[(this_type) & (df["Risk"] == "good")]
#         bad = df[(this_type) & (df["Risk"] == "bad")]
#         # score = -1 + len(good) / (len(bad) + 0.0001)
#         g, b = len(good), len(bad)
#         score = (g - b) / (g + b + 1e-9)
#         # print(f"Total with {column} [{i}]:  {len(df[this_type])}")
#         # print(f"    Good: {len(good)}")
#         # print(f"    Bad: {len(bad)}")
#         # print(f"    Points for this {column}: {score}\n")
#         if column not in keys:
#             keys[column] = {}
#         keys[column][option] = score    

The below code is the actual ai rule generation.

It uses an upgraded formula from the previous one (x - y) / (x + y)
This formula normalises the score between -1 and 1, or a positive or negative influence on the overall user score.
This is an improvement over the previous scoring method of  -1 + len(good) / len(bad), as it prevents something like 99 / 1 which would result in a score of 98, overpowering any other metric
All fields are iterated over, with their score calculated, and is stored in the keys dictionary

The rule is whether the score generated is above or below 0

In [ ]:
# Where scores are kept
keys = {}

# : means the start of the table, and we are making a new table based off (len(df) * 0.8), or the 80% of the dataframe from the start
# We only work on 80% so we can verify our model works with the final 20%
# The final 20% has an answer book built in so we can verify the result for each row, whereas the testing data doesn't
df_subset = df.iloc[:int(len(df) * 0.8)]

# print(len(df), len(df_subset), len(df) * 0.8, len(df) - len(df_subset))

# Go through every field in the dataset
for column in df_subset.columns:
    # Ignore ID and Risk, since ID should not contribute to the score, and Risk would refer to itself which makes no sense.
    if column == "ID" or column == "Risk": continue

    # Get every possible option for the column
    for option in df_subset[column].unique():
        # print(f"Column: {column} Option: {option}\n")

        # Create a table with every row where option X is found (For example, every row with '44' in the 'Age' column)
        this_type = df_subset[column] == option;
        # print(df_subset)

        # 'good' and 'bad' are tables of every row with their respective 'Risk' type. There may be 50 'good' and 23 'bad' rows for an age of 44
        good = df_subset[(this_type) & (df_subset["Risk"] == "good")]
        bad = df_subset[(this_type) & (df_subset["Risk"] == "bad")]

        # However many there are of each
        g, b = len(good), len(bad)

        # Normalised between -1 and 1. 1e-9 is a miniscule number to prevent division-by-zero errors
        score = (g - b) / (g + b + 1e-9)

        if column not in keys:
            keys[column] = {}
        
        # Add the score to keys
        keys[column][option] = score    

In [86]:
print(f"Score if you are a male: {keys["Sex"]["male"]:+.5f}")
print(f"Score if you are 22 years old: {keys["Age"][22]:+.5f}")
print(f"Score if you are 50 years old: {keys["Age"][50]:+.5f}")


Score if you are a male: +0.07107
Score if you are 22 years old: -0.22581
Score if you are 50 years old: +0.55556


In [74]:
# correct = 0
# incorrect = 0

# for index, row in df.iterrows():
#     user_score = 0
#     for column in df.columns:
#         # print(df.iloc[index][column])
#         if column not in keys:
#             continue
#         score_to_add = keys[column][df.iloc[index][column]]
#         user_score += score_to_add

#     is_correct = (user_score >= 0) == (row["Risk"] == "good")
#     correct += is_correct
    
#     print(f"Act. risk: {row["Risk"]}, calc. risk: {"good" if user_score >= 0 else "bad"} ({"correct" if is_correct else "incorrect"}), risk val.: {user_score:+.3f}")
# total_size = len(df.index)
# print(f"Correct: {correct}, incorrect: {total_size - correct}.")
# print(f"{correct / total_size * 100:.3f}% accuracy.")

The below code runs over the previous subset with the scoring system to test if the stored result aligns with the calculated result/

In [ ]:
# We compare to our 'answer book' with our scores and see if our result matches the 'Risk' column
correct = 0

for index, row in df_subset.iterrows():
    # The scores are added to the below variable
    user_score = 0

    # For every field for a given row, get the score and add it to the user_score
    for column in df_subset.columns:
        # print(df_subset.iloc[index][column])

        # Skips over ID / Risk
        if column not in keys: continue

        # Go into keys (for example, keys["Age"]) and extract the score for the particular value -> keys["Age"].get(44)
        score_to_add = keys[column].get(row[column], 0)
        user_score += score_to_add

    # The score represents a correct result if the XNOR gate below outputs TRUE
    # If the user_score >= 0 and the Risk row is "good" (TRUE == TRUE) then the result is TRUE
    # If the user_score < 0 and the Risk row is not "good" (FALSE == FALSE) then the result is TRUE
    # If either of these mismatch, (TRUE == FALSE) or (FALSE == TRUE), then the result is FALSE
    is_correct = (user_score >= 0) == (row["Risk"] == "good")

    # Since a bool is just 0 or 1, we can add is_correct to correct
    correct += is_correct
    
    # print(f"Act. risk: {row["Risk"]}, calc. risk: {"good" if user_score >= 0 else "bad"} ({"correct" if is_correct else "incorrect"}), risk val.: {user_score:+.3f}")
total_size = len(df_subset.index)
print(f"Total rows: {total_size}")
print(f"Correct: {correct}, incorrect: {total_size - correct}.")
print(f"{correct / total_size * 100:.3f}% accuracy.")

Total rows: 896
Correct: 876, incorrect: 20.
97.768% accuracy.


The above output should be as follows:

Total rows: 896
Correct: 876, incorrect: 20.
97.768% accuracy.

A score of ~98% is fairly good, however this is only for the data we tested on.

In [90]:
# We create a new subset, starting after 80% of the original dataframe (since the : is after)
# This new 20% subset can be used to test our AI rule with a cheatsheet, since it includes the Risk column
df_subset_tail = df.iloc[int(len(df) * 0.8):].reset_index(drop=True)
correct = 0

# This is the same deal as before, but ran on unknown data
for index, row in df_subset_tail.iterrows():
    user_score = 0
    for column in df_subset_tail.columns:
        if column not in keys:
            continue
        # score_to_add = keys[column][row[column]]
        score_to_add = keys[column].get(row[column], 0)
        user_score += score_to_add

    is_correct = (user_score >= 0) == (row["Risk"] == "good")
    correct += is_correct
    
    # print(f"Act. risk: {row["Risk"]}, calc. risk: {"good" if user_score >= 0 else "bad"} ({"correct" if is_correct else "incorrect"}), risk val.: {user_score:+.3f}")
total_size = len(df_subset_tail.index)
print(f"Total rows: {total_size}")
print(f"Correct: {correct}, incorrect: {total_size - correct}.")
print(f"{correct / total_size * 100:.3f}% accuracy.")

Total rows: 224
Correct: 166, incorrect: 58.
74.107% accuracy.


The result should be similar ot what follows:

Total rows: 224
Correct: 166, incorrect: 58.
74.107% accuracy.

This shows just under 3/4 correct answers for the data we tested on, which isn't terrible!

In [97]:
# tf for test frame (aren't I creative?). Read in the data from the test document into pandas
tf = pd.read_csv("test_data.csv")
rejected = 0

# We do the same as we have done when testing previous data sets:
# iterating over every column,
# adding every score found in keys for that field,
# outputting the final score

for index, row in tf.iterrows():
    user_score = 0
    for column in tf.columns:
        if column not in keys:
            continue
        # score_to_add = keys[column][df.iloc[index][column]]
        score_to_add = keys[column].get(row[column], 0)

        user_score += score_to_add

    rejected += user_score < 0
    
    print(f"Calculated risk for this individual: {"good" if user_score >= 0 else "bad"}, risk score evaluated as: {user_score:+.3f}")

total_size = len(df.index)
print(f"Training data 'bad' risk percent: {((total_size - len(df[df["Risk"] == "bad"])) / total_size) * 100:.1f}%")
total_size_test = len(tf.index)
print(f"Testing data 'bad' risk percent: {((rejected) / total_size_test) * 100:.1f}%")
print(f"good : bad -> {total_size_test - rejected} : {rejected}")


Calculated risk for this individual: bad, risk score evaluated as: -0.632
Calculated risk for this individual: good, risk score evaluated as: +1.065
Calculated risk for this individual: bad, risk score evaluated as: -0.162
Calculated risk for this individual: bad, risk score evaluated as: -0.812
Calculated risk for this individual: bad, risk score evaluated as: -0.981
Calculated risk for this individual: bad, risk score evaluated as: -1.436
Calculated risk for this individual: bad, risk score evaluated as: -0.104
Calculated risk for this individual: bad, risk score evaluated as: -0.946
Calculated risk for this individual: bad, risk score evaluated as: -1.085
Calculated risk for this individual: good, risk score evaluated as: +1.684
Calculated risk for this individual: good, risk score evaluated as: +0.154
Calculated risk for this individual: bad, risk score evaluated as: -1.263
Calculated risk for this individual: good, risk score evaluated as: +0.504
Calculated risk for this individua

The above results should show the following:

Training data 'bad' risk percent: 50.0%
Testing data 'bad' risk percent: 67.1%
good : bad -> 92 : 188

This means, according to our AI, ~67% have a Risk of 'bad'.
Additional factors with what contributes to a row's score can be explored with interesting connotations. For example, the below output shows a difference between the genders of an individual in their row.

In [91]:
print(f"Males have {keys["Sex"]["male"]:+.3f} ({len(df[(df["Sex"] == "male") & (df["Risk"] == "good")])} / {len(df[(df["Sex"] == "male")])} = {(len(df[(df["Sex"] == "male") & (df["Risk"] == "good")]) / len(df[(df["Sex"] == "male")])):.3f}).")
print(f"Females have {keys["Sex"]["female"]:+.3f} ({len(df[(df["Sex"] == "female") & (df["Risk"] == "good")])} / {len(df[(df["Sex"] == "female")])} = {(len(df[(df["Sex"] == "female") & (df["Risk"] == "good")]) / len(df[(df["Sex"] == "female")])):.3f}).")
print(f"That is a difference of {abs(keys["Sex"]["female"] - keys["Sex"]["male"]):.3f}")

Males have +0.071 (397 / 748 = 0.531).
Females have -0.127 (163 / 372 = 0.438).
That is a difference of 0.198


The result demonstrates a bias towards males and their Risk column. If we assume 'Risk' means that a loan may be granted to an individual, this shows a gender bias against female individuals.
It should be noticed this is an inherent result from the chosen method of deriving a rule for calculating risk. This isn't necessarily representative of the true scoring factor.